# RQ4: Does autonomy level predict safety rating and vehicle performance?

**Hypothesis:** Higher autopilot levels correlate with higher safety ratings and superior performance.

**Methodology:** Spearman correlation + one-way ANOVA → box plots per autopilot level (PDF) + summary table (CSV).

In [ ]:
import pandas as pd, numpy as np, os, warnings
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, f_oneway
from matplotlib.patches import Patch
warnings.filterwarnings("ignore")
plt.rcParams.update({"font.family":"serif","font.size":11,"axes.titlesize":12,"axes.labelsize":11,"figure.dpi":300,"axes.spines.top":False,"axes.spines.right":False})

for p in ["/kaggle/input/electric-vehicle-market-and-pricing-dataset-2026/ev_market_2026.csv","ev_market_2026.csv"]:
    if os.path.exists(p): df = pd.read_csv(p); break

df = df.dropna(subset=["autopilot_level","safety_rating","acceleration_0_60_mph","top_speed_mph"])
print("Shape:", df.shape)
levels = sorted(df["autopilot_level"].unique())
print("Autopilot levels:", levels)

In [ ]:
for t in ["safety_rating","acceleration_0_60_mph","top_speed_mph"]:
    rho, p = spearmanr(df["autopilot_level"], df[t])
    print(f"Autopilot vs {t}: rho={rho:.3f}, p={p:.4f}")

groups = [df[df["autopilot_level"]==l]["safety_rating"].values for l in levels]
F, p_anova = f_oneway(*groups)
print(f"
Safety Rating ANOVA: F={F:.2f}, p={p_anova:.4f}")

In [ ]:
COLORS = {0:"#aec6cf", 1:"#77b5d4", 2:"#3990c0", 3:"#1a5f8e"}
level_colors = [COLORS[l] for l in levels]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))

data1 = [df[df["autopilot_level"]==l]["safety_rating"].values for l in levels]
bp1 = ax1.boxplot(data1, patch_artist=True, widths=0.55, medianprops={"color":"#333","linewidth":1.8}, whiskerprops={"linewidth":1.2}, capprops={"linewidth":1.2})
for patch, color in zip(bp1["boxes"], level_colors): patch.set_facecolor(color); patch.set_alpha(0.75)
ax1.set_xticklabels([f"Level {l}" for l in levels])
ax1.set_xlabel("Autopilot Level"); ax1.set_ylabel("Safety Rating (stars)")
ax1.set_title(f"Safety Rating by Autopilot Level\n(F={F:.2f}, p={p_anova:.4f})", fontsize=11)

data2 = [df[df["autopilot_level"]==l]["acceleration_0_60_mph"].values for l in levels]
F2, p2 = f_oneway(*data2)
bp2 = ax2.boxplot(data2, patch_artist=True, widths=0.55, medianprops={"color":"#333","linewidth":1.8}, whiskerprops={"linewidth":1.2}, capprops={"linewidth":1.2})
for patch, color in zip(bp2["boxes"], level_colors): patch.set_facecolor(color); patch.set_alpha(0.75)
ax2.set_xticklabels([f"Level {l}" for l in levels])
ax2.set_xlabel("Autopilot Level"); ax2.set_ylabel("0-60 mph (s) — lower = faster")
ax2.set_title(f"Acceleration by Autopilot Level\n(F={F2:.2f}, p={p2:.4f})", fontsize=11)

leg = [Patch(facecolor=COLORS[l], alpha=0.75, label=f"Level {l}") for l in levels]
fig.legend(handles=leg, loc="lower center", ncol=4, frameon=False, fontsize=10)
fig.suptitle("EV Safety & Performance by Autonomy Level", fontsize=13, y=1.01)
plt.tight_layout(rect=[0,0.06,1,1])
fig.savefig("RQ4_Autopilot_Safety_Performance.pdf", bbox_inches="tight", format="pdf")
plt.show(); print("Saved: RQ4_Autopilot_Safety_Performance.pdf")

In [ ]:
rows = []
for l in levels:
    sub = df[df["autopilot_level"]==l]
    rows.append({"Autopilot Level":l,"N":len(sub),"Mean Safety Rating":round(sub["safety_rating"].mean(),2),"SD Safety":round(sub["safety_rating"].std(),2),"Mean 0-60 (s)":round(sub["acceleration_0_60_mph"].mean(),2),"SD 0-60":round(sub["acceleration_0_60_mph"].std(),2),"Mean Top Speed (mph)":round(sub["top_speed_mph"].mean(),1)})
tbl = pd.DataFrame(rows)
tbl.to_csv("RQ4_Summary_Table.csv", index=False)
print("Saved: RQ4_Summary_Table.csv"); tbl